# 03 Explainability and Evaluation

Final-day notebook for Person C. Run Person A preprocessing, Person A trust pipeline, and Person B model verification before running explainability outputs.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath('..'))

import pandas as pd
from shared import config
from evaluation.model_evaluation import (
    evaluate_predictions,
    save_metrics,
    save_classification_report,
    save_predictions,
    save_confusion_matrix_plot,
)
from evaluation.fairness_metrics import (
    evaluate_fairness,
    prediction_distribution,
    save_fairness_results,
    save_prediction_distribution,
)
from explainability.shap_explainer import split_correct_and_misclassified

print('Imports ready')

In [ ]:
required_files = [
    config.PROCESSED_HATE_SPEECH_PATH,
    config.PROCESSED_HATEXPLAIN_PATH,
    config.TRAINING_DATASET_PATH,
    config.PROCESSED_ANNOTATOR_WEIGHTS_PATH,
    config.PROCESSED_SOFT_LABELS_PATH,
]

for path in required_files:
    print(f'{path}:', 'FOUND' if Path(path).exists() else 'MISSING')

In [ ]:
dataset_path = config.FINAL_TRAINING_DATASET_PATH if config.FINAL_TRAINING_DATASET_PATH.exists() else config.TRAINING_DATASET_PATH
training_df = pd.read_csv(dataset_path)
print('Training dataset loaded:', training_df.shape)
training_df.head()

## Model Evaluation

Replace `model_prediction` with Person B's inference output when the trained model is available. If predictions are already present, the notebook saves metrics, predictions, classification report and confusion matrix.

In [ ]:
true_col = 'predicted_label'
pred_col = 'model_prediction' if 'model_prediction' in training_df.columns else 'predicted_label'

metrics, report, matrix = evaluate_predictions(training_df, true_col, pred_col)
labels = sorted(training_df[true_col].dropna().unique())

save_metrics(metrics)
save_classification_report(training_df[true_col], training_df[pred_col])
save_predictions(training_df, training_df[true_col], training_df[pred_col])
save_confusion_matrix_plot(matrix, labels)

metrics

## Fairness Evaluation

Fairness metrics are calculated when protected attribute columns are available in the dataset.

In [ ]:
fairness_results = evaluate_fairness(training_df, true_col, pred_col)
distribution = prediction_distribution(training_df, pred_col)

save_fairness_results(fairness_results)
save_prediction_distribution(distribution)

fairness_results.head()

## Correct and Misclassified Samples

These subsets can be passed into SHAP and Captum once the trained model and inference function are available.

In [ ]:
predictions_df = training_df.copy()
predictions_df['true_label'] = training_df[true_col]
predictions_df['predicted_label'] = training_df[pred_col]
correct_samples, misclassified_samples = split_correct_and_misclassified(predictions_df)

print('Correct samples:', len(correct_samples))
print('Misclassified samples:', len(misclassified_samples))